In [ ]:
!pip install scikit-learn
!pip install tensorflow
!pip install torch

In [1]:
import os
import json
import numpy as np
import pandas as pd
import torch
import time
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf

In [2]:
df = pd.read_json("C:/Users/007pe/Downloads/ready_data.json", lines=True)

In [ ]:
print(df)

In [3]:
df_copy2 = df.copy(deep=True)
df_copy2 = df_copy2[df_copy2['Speaker_party_name'].apply(lambda x: x is not None)]

In [4]:
from collections import defaultdict
label_to_broad_group = {
    'Centre-right to right-wing': 'Centre-right to right-wing',
    'Right-wing': 'Centre-right to right-wing',
    'Right-wing to far-right': 'Centre-right to right-wing',
    'Centre-left': 'Centre-left',
    'Centre to centre-left': 'Centre-left',
    'Centre-left to left-wing': 'Centre-left',
    'Left-wing': 'Left-wing to far-left',
    'Left-wing to far-left': 'Left-wing to far-left',
    'Non-partisan': 'Non-partisan',
    'Cleric': 'Non-partisan',
    'Independent': 'Independent'
}
mapped_dict = defaultdict(set)

for party, ideology in label_to_broad_group.items():
    mapped_dict[ideology].update([party])

def map_values(category):
    for ideology, parties in mapped_dict.items():
        if category in parties:
            return ideology
    return None

# df_copy2['Speaker_party_name'] = df_copy['Speaker_party_name'].apply(map_values)
print(df_copy2)
print(df_copy2['Speaker_party_name'].value_counts())

                Speaker_party_name  \
0       Centre-right to right-wing   
1            Centre to centre-left   
2       Centre-right to right-wing   
3                      Centre-left   
4       Centre-right to right-wing   
...                            ...   
591683                 Centre-left   
591684  Centre-right to right-wing   
591685                 Centre-left   
591686  Centre-right to right-wing   
591687                 Centre-left   

                                                embedding  
0       [-0.017255421700000002, 0.041122213000000005, ...  
1       [-0.0191462822, 0.013131545900000001, -0.12457...  
2       [-0.1000296474, -0.0525700152, 0.0945084095, 0...  
3       [0.0851034224, 0.014632830400000001, -0.022017...  
4       [-0.0032701697, 0.11290641870000001, 0.1563548...  
...                                                   ...  
591683  [-0.1648578346, -0.1362403184, 0.0647174343, -...  
591684  [-0.1161431968, -0.0108349603, 0.143547073, 0....  
591

In [ ]:
encoder = LabelEncoder()
y = encoder.fit_transform(y)  # Convert labels to integers
num_classes = len(encoder.classes_) 

In [ ]:
print(num_classes)

In [9]:
X = np.vstack(df_copy2['embedding'].values)
y = df_copy2['Speaker_party_name']

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=0, stratify=y)

In [15]:
model = Sequential([
    Input(shape=(X_train.shape[1],)),  # Explicit Input layer
    Dense(64, activation='relu'),      # Hidden layer
    Dropout(0.5),                      # Dropout for regularization
    Dense(num_classes, activation='softmax')  # Output layer for multi-class classification
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [ ]:
model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_test, y_test))

# Evaluate the model
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {accuracy * 100:.2f}%")

In [56]:
def create_model(hidden_units=256, dropout_rate=0.4, learning_rate=0.001):
    model = Sequential([
        Input(shape=(X_train.shape[1],)),  # Input layer
        Dense(hidden_units, activation='relu'),  # Hidden layer
        Dropout(dropout_rate),
        Dense(int(hidden_units), activation='relu'),  # Hidden layer
        Dropout(dropout_rate),
        Dense(int(hidden_units), activation='relu'),  # Hidden layer
        Dropout(dropout_rate),# Dropout for regularization
        Dense(num_classes, activation='softmax')  # Output layer
    ])
    
    # Compile the model
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model
    
seq_model = create_model()

In [57]:
seq_model.fit(X_train, y_train, epochs=10, validation_data=(X_test, y_test))

Epoch 1/10
14771/14771 ━━━━━━━━━━━━━━━━━━━━ 35s 2ms/step - accuracy: 0.6824 - loss: 0.7234 - val_accuracy: 0.7111 - val_loss: 0.6594
Epoch 2/10
14771/14771 ━━━━━━━━━━━━━━━━━━━━ 36s 2ms/step - accuracy: 0.7073 - loss: 0.6708 - val_accuracy: 0.7145 - val_loss: 0.6512
Epoch 3/10
14771/14771 ━━━━━━━━━━━━━━━━━━━━ 33s 2ms/step - accuracy: 0.7138 - loss: 0.6580 - val_accuracy: 0.7191 - val_loss: 0.6428
Epoch 4/10
14771/14771 ━━━━━━━━━━━━━━━━━━━━ 33s 2ms/step - accuracy: 0.7177 - loss: 0.6505 - val_accuracy: 0.7200 - val_loss: 0.6396
Epoch 5/10
14771/14771 ━━━━━━━━━━━━━━━━━━━━ 36s 2ms/step - accuracy: 0.7206 - loss: 0.6442 - val_accuracy: 0.7199 - val_loss: 0.6411
Epoch 6/10
14771/14771 ━━━━━━━━━━━━━━━━━━━━ 33s 2ms/step - accuracy: 0.7214 - loss: 0.6427 - val_accuracy: 0.7245 - val_loss: 0.6347
Epoch 7/10
14771/14771 ━━━━━━━━━━━━━━━━━━━━ 36s 2ms/step - accuracy: 0.7254 - loss: 0.6366 - val_accuracy: 0.7234 - val_loss: 0.6322
Epoch 8/10
14771/14771 ━━━━━━━━━━━━━━━━━━━━ 35s 2ms/step - accuracy: 

In [52]:
import keras_tuner as kt

def build_model(hp):
    # Define hyperparameters to tune
    hidden_units = hp.Int('hidden_units', min_value=32, max_value=256, step=32)
    dropout_rate = hp.Float('dropout_rate', min_value=0.2, max_value=0.5, step=0.1)
    learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4, 1e-5])
    
    # Create the model
    model = create_model(hidden_units=hidden_units, dropout_rate=dropout_rate, learning_rate=learning_rate)
    return model

# Initialize the tuner
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=20,  # Number of hyperparameter combinations to try
    executions_per_trial=2,  # Number of models to train per trial
    directory='my_tuning_dir',  # Directory to save results
    project_name='text_classification'
)

# Perform the search
tuner.search(X_train, y_train, epochs=20, validation_data=(X_test, y_test))

# Get the best hyperparameters
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print(f"""
Best hyperparameters:
- Hidden Units: {best_hps.get('hidden_units')}
- Dropout Rate: {best_hps.get('dropout_rate')}
- Learning Rate: {best_hps.get('learning_rate')}
""")

# Train the final model with the best hyperparameters
best_model = tuner.hypermodel.build(best_hps)
history = best_model.fit(X_train, y_train, epochs=50, validation_data=(X_test, y_test))

Trial 20 Complete [00h 15m 21s]
val_accuracy: 0.7263552844524384

Best val_accuracy So Far: 0.7330450415611267
Total elapsed time: 06h 39m 10s

Best hyperparameters:
- Hidden Units: 224
- Dropout Rate: 0.4
- Learning Rate: 0.0001

Epoch 1/50
14771/14771 ━━━━━━━━━━━━━━━━━━━━ 34s 2ms/step - accuracy: 0.6450 - loss: 0.7986 - val_accuracy: 0.7026 - val_loss: 0.6775
Epoch 2/50
14771/14771 ━━━━━━━━━━━━━━━━━━━━ 32s 2ms/step - accuracy: 0.7011 - loss: 0.6890 - val_accuracy: 0.7112 - val_loss: 0.6601
Epoch 3/50
14771/14771 ━━━━━━━━━━━━━━━━━━━━ 32s 2ms/step - accuracy: 0.7078 - loss: 0.6728 - val_accuracy: 0.7151 - val_loss: 0.6502
Epoch 4/50
14771/14771 ━━━━━━━━━━━━━━━━━━━━ 32s 2ms/step - accuracy: 0.7146 - loss: 0.6593 - val_accuracy: 0.7179 - val_loss: 0.6426
Epoch 5/50
14771/14771 ━━━━━━━━━━━━━━━━━━━━ 33s 2ms/step - accuracy: 0.7178 - loss: 0.6521 - val_accuracy: 0.7205 - val_loss: 0.6388
Epoch 6/50
14771/14771 ━━━━━━━━━━━━━━━━━━━━ 32s 2ms/step - accuracy: 0.7196 - loss: 0.6467 - val_accurac

In [23]:
loss, accuracy = best_model.evaluate(X_test, y_test)
print(f"Test Accuracy: {accuracy * 100:.2f}%")

3693/3693 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - accuracy: 0.6670 - loss: 0.9362
Test Accuracy: 66.65%


In [8]:
%%time
def train_model_with_balanced_data():
    """
    Example of how to use the balanced split with a classifier.
    """

    from sklearn.linear_model import LogisticRegression
    clf = LogisticRegression(C=0.1, penalty='l1', solver='liblinear', max_iter=500)
    clf.fit(X_train, y_train)
    
    # Evaluate
    from sklearn.metrics import classification_report
    y_pred = clf.predict(X_test)
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, zero_division=0))
    
    return clf

clf = train_model_with_balanced_data()


Classification Report:
                            precision    recall  f1-score   support

     Centre to centre-left       0.21      0.00      0.01      6893
               Centre-left       0.49      0.40      0.44     28134
  Centre-left to left-wing       0.58      0.08      0.14      6123
Centre-right to right-wing       0.68      0.92      0.78     69764
                    Cleric       0.43      0.01      0.01       456
               Independent       0.00      0.00      0.00       435
                 Left-wing       1.00      0.00      0.00       515
     Left-wing to far-left       0.00      0.00      0.00         3
              Non-partisan       0.26      0.02      0.03      4155
                Right-wing       0.41      0.04      0.08      1607
   Right-wing to far-right       0.00      0.00      0.00        81

                  accuracy                           0.64    118166
                 macro avg       0.37      0.13      0.14    118166
              weighted

In [ ]:
log_reg = LogisticRegression()
log_reg.fit(X_train, y_train)
param_grid = [
    {'penalty':['l1','l2','elasticnet','none'],
    'C' : np.logspace(-2,2,5),
    'solver': ['lbfgs','newton-cg','liblinear','sag','saga'],
    'max_iter'  : [5000]
}
]

clf = GridSearchCV(log_reg, param_grid = param_grid, cv = 3, verbose=4 ,n_jobs=3)
clf

In [ ]:
svm_classifier = SVC(kernel='linear')
svm_classifier.fit(X_train, y_train)

svm_classifier_path = '/content/drive/MyDrive/Colab/svm_classifier_model.pkl'
joblib.dump(svm_classifier, svm_classifier_path)

y_pred = svm_classifier.predict(X_test)

print(classification_report(y_test, y_pred))

In [ ]:
rfm_classifier = RandomForestClassifier(n_estimators=100, n_jobs=3, verbose=3, random_state=0)

rfm_classifier.fit(X_train, y_train)

y_pred = rfm_classifier.predict(X_test)
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
pip install matplotlib

In [22]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV, cross_val_score
from sklearn.metrics import make_scorer, accuracy_score, f1_score, precision_score, recall_score

def tune_random_forest(X_train, y_train, cv=5, n_iter=100, verbose=3):
    """
    Tune hyperparameters for a Random Forest model using RandomizedSearchCV
    
    Parameters:
    -----------
    X_train : array-like
        The feature matrix of training data (embedded speeches)
    y_train : array-like
        Target variable for training data
    cv : int
        Number of cross-validation folds
    n_iter : int
        Number of parameter settings sampled in RandomizedSearchCV
    verbose : int
        Verbosity level
        
    Returns:
    --------
    best_model : RandomForestClassifier
        The best model found
    search : RandomizedSearchCV
        The complete search results
    """
    
    # Define the parameter space
    param_dist = {
        'n_estimators': [50, 100, 200],  # Fewer estimators
        'max_depth': [10, 20, None],  # Limited depth options
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'max_features': ['sqrt', 'log2'],  # Only use built-in options
        'bootstrap': [True],  # Default only
        # Removed class_weight to save memory
        'criterion': ['gini', 'entropy']
    }
    
    # Define a scoring dictionary - useful for multi-metric evaluation
    scoring = {
        'accuracy': make_scorer(accuracy_score),
        'f1_weighted': make_scorer(f1_score, average='weighted'),
        'precision': make_scorer(precision_score, average='weighted'),
        'recall': make_scorer(recall_score, average='weighted')
    }
    
    # Create a base model
    rf = RandomForestClassifier(random_state=0, n_jobs=3)
    
    # Setup the randomized search with cross-validation
    search = RandomizedSearchCV(
        estimator=rf,
        param_distributions=param_dist,
        n_iter=n_iter,
        cv=cv,
        verbose=verbose,
        random_state=0,
        n_jobs=3,
        scoring=scoring,
        refit='f1_weighted'  # You can change this to the metric you want to optimize
    )
    
    # Fit the randomized search
    search.fit(X_train, y_train)
    
    # Get the best model
    best_model = search.best_estimator_
    
    # Print the best parameters
    print("Best Parameters:", search.best_params_)
    print(f"Best Score ({search.refit}): {search.best_score_:.4f}")
    
    # Print all scores for the best parameters
    print("Scores for best parameters:")
    for metric, score in search.cv_results_['mean_test_score'][search.best_index_].items():
        print(f"{metric}: {score:.4f}")
    
    return best_model, search

def refine_with_grid_search(X_train, y_train, best_params, cv=5, verbose=3):
    """
    Refine the best parameters from RandomizedSearchCV using GridSearchCV
    
    Parameters:
    -----------
    X_train : array-like
        The feature matrix of training data
    y_train : array-like
        Target variable for training data
    best_params : dict
        Best parameters from RandomizedSearchCV
    cv : int
        Number of cross-validation folds
    verbose : int
        Verbosity level
        
    Returns:
    --------
    best_model : RandomForestClassifier
        The best model found after refinement
    grid_search : GridSearchCV
        The complete grid search results
    """
    
    # Create narrower parameter ranges around the best parameters
    param_grid = {}
    
    # For n_estimators
    best_n_est = best_params['n_estimators']
    param_grid['n_estimators'] = [max(best_n_est - 100, 50), best_n_est, min(best_n_est + 100, 1000)]
    
    # For max_depth
    if best_params['max_depth'] is None:
        param_grid['max_depth'] = [None]
    else:
        best_depth = best_params['max_depth']
        param_grid['max_depth'] = [max(best_depth - 10, 5), best_depth, min(best_depth + 10, 150)]
    
    # For min_samples_split
    best_split = best_params['min_samples_split']
    param_grid['min_samples_split'] = [max(best_split - 1, 2), best_split, min(best_split + 1, 20)]
    
    # For min_samples_leaf
    best_leaf = best_params['min_samples_leaf']
    param_grid['min_samples_leaf'] = [max(best_leaf - 1, 1), best_leaf, min(best_leaf + 1, 10)]
    
    # Keep the best values for other parameters
    param_grid['max_features'] = [best_params['max_features']]
    param_grid['bootstrap'] = [best_params['bootstrap']]
    param_grid['class_weight'] = [best_params['class_weight']]
    param_grid['criterion'] = [best_params['criterion']]
    
    # Define scoring - same as before
    scoring = {
        'accuracy': make_scorer(accuracy_score),
        'f1_weighted': make_scorer(f1_score, average='weighted'),
        'precision': make_scorer(precision_score, average='weighted'),
        'recall': make_scorer(recall_score, average='weighted')
    }
    
    # Create a base model with the best parameters
    rf = RandomForestClassifier(random_state=0, n_jobs=3, **best_params)
    
    # Setup the grid search
    grid_search = GridSearchCV(
        estimator=rf,
        param_grid=param_grid,
        cv=cv,
        verbose=verbose,
        n_jobs=3,
        scoring=scoring,
        refit='f1_weighted'  # Same as before
    )
    
    # Fit the grid search
    grid_search.fit(X_train, y_train)
    
    # Get the best model
    best_model = grid_search.best_estimator_
    
    # Print the best parameters
    print("Refined Best Parameters:", grid_search.best_params_)
    print(f"Refined Best Score ({grid_search.refit}): {grid_search.best_score_:.4f}")
    
    return best_model, grid_search

def evaluate_final_model(model, X_test, y_test):
    """
    Evaluate the final model on the test set
    
    Parameters:
    -----------
    model : RandomForestClassifier
        The trained model
    X_test : array-like
        The feature matrix of test data
    y_test : array-like
        Target variable for test data
        
    Returns:
    --------
    results : dict
        Dictionary containing evaluation metrics
    """
    # Make predictions
    y_pred = model.predict(X_test)
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    
    # Create results dictionary
    results = {
        'accuracy': accuracy,
        'f1_weighted': f1,
        'precision': precision,
        'recall': recall
    }
    
    # Print results
    print("Test Set Evaluation:")
    for metric, value in results.items():
        print(f"{metric}: {value:.4f}")
    
    return results

def analyze_feature_importance(model, feature_names=None):
    """
    Analyze feature importance from the trained model
    
    Parameters:
    -----------
    model : RandomForestClassifier
        The trained model
    feature_names : list, optional
        List of feature names
        
    Returns:
    --------
    importance_df : DataFrame
        DataFrame containing feature importance
    """
    # Get feature importances
    importances = model.feature_importances_
    
    # Create feature names if not provided
    if feature_names is None:
        feature_names = [f'feature_{i}' for i in range(len(importances))]
    
    # Create DataFrame
    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': importances
    })
    
    # Sort by importance
    importance_df = importance_df.sort_values('importance', ascending=False)
    
    # Print top 20 features
    print("Top 20 Important Features:")
    print(importance_df.head(20))
    
    return importance_df

In [ ]:
# Step 1: Run the initial randomized search
best_model, search_results = tune_random_forest(
    X_train, 
    y_train, 
    cv=2,  # 5-fold cross-validation
    n_iter=50  # Try 100 different combinations
)

# Step 2: Refine using grid search around the best parameters
refined_model, grid_results = refine_with_grid_search(
    X_train,
    y_train,
    search_results.best_params_,
    cv=5
)

# Step 3: Evaluate the final model
final_results = evaluate_final_model(refined_model, X_test, y_test)

# Step 4: Analyze feature importance
importance_df = analyze_feature_importance(refined_model)

Fitting 2 folds for each of 50 candidates, totalling 100 fits


In [ ]:
# Step 1: Run the initial randomized search
best_model, search_results = tune_random_forest(
    X_train, 
    y_train, 
    cv=5,  # 5-fold cross-validation
    n_iter=50  # Try 100 different combinations
)

# Step 2: Refine using grid search around the best parameters
refined_model, grid_results = refine_with_grid_search(
    X_train,
    y_train,
    search_results.best_params_,
    cv=5
)

# Step 3: Evaluate the final model
final_results = evaluate_final_model(refined_model, X_test, y_test)

# Step 4: Analyze feature importance
importance_df = analyze_feature_importance(refined_model)